# Хранение, сжатие и партиционирование — 30 заданий

Модуль сравнивает физические варианты на одинаковых данных. Все объекты создаются в `m_razhin`; источник `greenplum_training.storage_source` восстанавливается и не используется другими проектами.

## Результаты обучения

После **Хранение и партиционирование** вы должны объяснять физическое выполнение на coordinator/segments, связывать logical SQL с Motion/I/O/skew, выбирать дизайн по workload и доказывать решение измерениями.

## Ментальная модель

Heap, AO row и AOCO оптимальны для разных write/read patterns. Compression и column orientation уменьшают I/O, partition pruning ограничивает физические дочерние таблицы.

```text
client → coordinator (parse/optimize)
              │ dispatch slices
       ┌──────┼──────┐
       ▼      ▼      ▼
    segment segment segment
       └── Motion/interconnect ──┘
              │
              ▼
          coordinator
```
Coordinator не должен становиться местом обработки всех строк. Хороший план оставляет
scan/aggregate на сегментах и перемещает только необходимое.

## Данные и grain

Metrica по дате и учебные AO/AOCO таблицы. Полные схемы находятся в `data-catalog`. Общие external/raw объекты читаются, учебные результаты создаются только в `m_razhin`.

## Инженерный алгоритм

1. Назовите grain и ключ. 2. Оцените объём/cardinality. 3. Выберите distribution/storage/partition. 4. Предскажите Motion и I/O. 5. Создайте минимальный объект. 6. ANALYZE. 7. Снимите EXPLAIN и сегментные метрики. 8. Сверьте результат.

Выберите storage по workload, partition key по регулярному фильтру, разумную гранулярность и докажите pruning через EXPLAIN.

## Типичные ошибки

- Переносить правила PostgreSQL без учёта MPP.
- Выбирать distribution key только по высокой cardinality.
- Путать partitioning с distribution.
- Считать Broadcast всегда плохим, а Redistribute всегда допустимым.
- Сравнивать время единственного запуска без rows/Motion/I/O.
- Создавать external object с путём, доступным Windows, но не сегментам.

## Вопросы для самопроверки

1. Где физически лежит строка? 2. Какие slices выполнят сегменты? 3. Что и сколько передаёт Motion? 4. Как проявится skew? 5. Что произойдёт при повторной загрузке? 6. Как доказать результат из независимого источника?

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 100
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

## 1. Логическая и физическая модель

DDL определяет не только колонки. Для Greenplum важны четыре независимых решения: distribution policy, способ хранения, compression и partitioning. Один хороший параметр не компенсирует другой: сжатая таблица всё равно может иметь skew, а идеально распределённая — читать лишние партиции.

## 2. Heap

Heap наследует модель PostgreSQL: строки размещаются построчно, UPDATE/DELETE естественно создают новые версии MVCC, возможны индексы. Heap удобен для небольших изменяемых таблиц и справочников. Для большого append-only факта он часто уступает AO по последовательной загрузке, сжатию и аналитическому чтению.

### MVCC и обслуживание heap

UPDATE фактически создаёт новую версию, DELETE помечает старую невидимой. Старые версии занимают место до vacuum. Массовый ежедневный факт обычно лучше загружать новыми порциями, чем постоянно обновлять каждую строку.

## 3. Append-optimized row

AO row хранит последовательности добавленных строк и оптимизирован для bulk load и scan. Он не означает, что SQL-команда `UPDATE` синтаксически невозможна, но частые мелкие изменения противоречат назначению. AO row полезен, когда запрос обычно читает большую часть колонок.

## 4. Append-optimized column

AO column хранит колонки раздельно по row groups. Запрос, выбирающий пять колонок из пятидесяти, читает меньше данных. Однотипные значения лучше сжимаются. Цена — невыгодное точечное чтение всей строки и сложность частых изменений.

### Проекция колонок

Колоночное хранение помогает только если запрос действительно выбирает часть колонок. `SELECT *` заставляет прочитать все column streams. При проектировании витрины учитывайте реальные SELECT-листы аналитиков.

## 5. Compression

Сжатие уменьшает I/O и иногда ускоряет запрос, хотя требует CPU. `zlib` обычно сжимает сильнее, `zstd` даёт хороший баланс скорости и размера, RLE_TYPE эффективно кодирует длинные серии одинаковых значений. Результат зависит от порядка строк, типов и данных — алгоритм выбирают измерением.

### Compress level

Более высокий уровень не гарантирует лучший end-to-end результат. Дополнительный CPU может превышать экономию чтения. Сравнивайте размер, время загрузки и несколько типовых запросов после прогрева.

## 6. Измерение размера

`pg_relation_size` показывает основной физический объект, `pg_total_relation_size` включает дополнительные структуры. У partitioned parent почти нет пользовательских строк: размер нужно суммировать по leaf partitions. В Greenplum размер следует понимать как сумму файлов всех сегментов.

## 7. Partitioning

Partitioning делит одну логическую таблицу на физические leaf-таблицы по правилу. Основная польза — partition pruning и управление жизненным циклом: быстро добавить, обменять, очистить или удалить период.

### Range, list и default

Range подходит датам и числовым интервалам; обычно используется полуинтервал `[start,end)`. List подходит небольшому фиксированному набору категорий. Default принимает всё, что не соответствует явным частям, но может скрывать ошибку маршрутизации.

### Сколько партиций

Слишком крупная partition читает лишние данные и неудобна для обслуживания. Слишком мелкие части увеличивают каталоги, время планирования и число файлов. Гранулярность выбирают по фильтрам, объёму периода и операциям загрузки/удаления.

## 8. Partition pruning

Optimizer исключает leaf partitions, границы которых несовместимы с предикатом. Наиболее надёжен прямой sargable-фильтр по partition key. Функция, cast или сложное выражение над ключом может ограничить статический pruning.

In [ ]:
%%sql
EXPLAIN SELECT count(*)
FROM m_razhin.gps_11_range_partition
WHERE event_date>=DATE '2025-02-01' AND event_date<DATE '2025-03-01';

## 9. Direct Partition Exchange

Exchange меняет привязку физической таблицы к partition metadata. Это позволяет почти мгновенно заменить большой период: данные заранее готовятся и проверяются в staging, затем staging и leaf меняются местами без обычной перезаписи всех строк.

### Совместимость staging

Колонки, типы, порядок, distribution policy и storage options должны быть совместимы. `LIKE` уменьшает риск расхождения, но не заменяет проверку диапазона данных. Staging не должна содержать строки вне target boundaries.

### WITH/WITHOUT VALIDATION

С validation СУБД проверяет границы, что безопаснее, но может просканировать staging. `WITHOUT VALIDATION` быстрее и переносит ответственность на ETL. Перед ним обязательны собственные count, min/max key, NULL и quality checks.

### Безопасная последовательность exchange

1. Создать совместимую staging. 2. Загрузить данные. 3. Выполнить quality checks. 4. Добавить target partition. 5. Exchange. 6. Проверить counts и границы через parent. 7. Решить судьбу бывшей partition, оказавшейся staging-таблицей.

## 10. Типичные ошибки

- сравнивать размеры таблиц с разным числом строк;
- считать только parent partitioned table;
- выбирать compression по одному запросу;
- путать distribution key и partition key;
- использовать `SELECT *` и ждать выигрыша column orientation;
- выполнять exchange до проверки диапазона;
- создавать тысячи крошечных partitions.

## 11. Порядок практики

Сначала создавайте варианты на одном источнике, проверяйте count и policy, выполняйте ANALYZE, затем измеряйте размер и план. Для partition DDL всегда проверяйте дерево до и после операции. Не выполняйте опасные команды на `dds` или рабочей таблице задания.

### Задание 1. `m_razhin.gps_01_heap`

**Что сделать:** Создайте heap-копию `storage_source`, распределённую по event_id.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Heap задаётся отсутствием appendoptimized=true.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_01_heap здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',1);

### Задание 2. `m_razhin.gps_02_ao_row`

**Что сделать:** Создайте append-optimized row table без сжатия.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

WITH (appendoptimized=true, orientation=row).

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_02_ao_row здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',2);

### Задание 3. `m_razhin.gps_03_ao_column`

**Что сделать:** Создайте append-optimized column table без сжатия.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

orientation=column.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_03_ao_column здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',3);

### Задание 4. `m_razhin.gps_04_ao_zlib`

**Что сделать:** Создайте AO column с zlib и уровнем 5.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

compresstype=zlib, compresslevel=5.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_04_ao_zlib здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',4);

### Задание 5. `m_razhin.gps_05_ao_zstd`

**Что сделать:** Создайте AO column с zstd и уровнем 5.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Проверьте поддержку алгоритма данным кластером.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_05_ao_zstd здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',5);

### Задание 6. `m_razhin.gps_06_ao_rle`

**Что сделать:** Создайте AO column с RLE_TYPE для подходящих колонок.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

RLE особенно полезен повторяющимся значениям.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_06_ao_rle здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',6);

### Задание 7. `m_razhin.gps_07_storage_catalog`

**Что сделать:** Создайте VIEW параметров хранения таблиц 01–06.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Используйте pg_appendonly и pg_class.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_07_storage_catalog здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',7);

### Задание 8. `m_razhin.gps_08_table_sizes`

**Что сделать:** Создайте VIEW total/heap/index/ao размера каждого варианта.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Сравнивайте одинаковое число строк.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_08_table_sizes здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',8);

### Задание 9. `m_razhin.gps_09_compression_ratio`

**Что сделать:** Создайте VIEW коэффициента размера относительно heap.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

ratio = heap_size / variant_size.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_09_compression_ratio здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',9);

### Задание 10. `m_razhin.gps_10_scan_benchmark`

**Что сделать:** Создайте таблицу результатов времени/плана чтения нескольких колонок.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Не делайте вывод по единственному случайному запуску.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_10_scan_benchmark здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',10);

## Уровень 2 — partitioning и pruning

### Задание 11. `m_razhin.gps_11_range_partition`

**Что сделать:** Создайте range-partitioned таблицу по event_date с месячными партициями.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Исходник содержит 90 дней.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_11_range_partition здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',11);

### Задание 12. `m_razhin.gps_12_partition_catalog`

**Что сделать:** Создайте VIEW дерева партиций, границ и физических имён.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Используйте системные представления partition metadata.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_12_partition_catalog здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',12);

### Задание 13. `m_razhin.gps_13_partition_counts`

**Что сделать:** Создайте VIEW количества строк в каждой leaf partition.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Считать нужно физические leaf-таблицы.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_13_partition_counts здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',13);

### Задание 14. `m_razhin.gps_14_pruning_plan`

**Что сделать:** Сохраните число просканированных partitions для фильтра одного месяца.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Проверяйте EXPLAIN.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_14_pruning_plan здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',14);

### Задание 15. `m_razhin.gps_15_no_pruning`

**Что сделать:** Покажите число partitions при выражении, мешающем pruning.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Сравните с sargable-предикатом.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_15_no_pruning здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',15);

### Задание 16. `m_razhin.gps_16_default_partition`

**Что сделать:** Создайте таблицу с default partition и загрузите строку вне диапазона.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Default принимает данные без подходящей явной части.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_16_default_partition здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',16);

### Задание 17. `m_razhin.gps_17_add_partition`

**Что сделать:** Добавьте новую месячную partition и загрузите строки этого месяца.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Границы не должны пересекаться.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_17_add_partition здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',17);

### Задание 18. `m_razhin.gps_18_drop_partition`

**Что сделать:** На учебной копии удалите старую partition и подтвердите изменение дерева.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

DROP PARTITION удаляет данные части.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_18_drop_partition здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',18);

### Задание 19. `m_razhin.gps_19_truncate_partition`

**Что сделать:** Очистите одну partition без изменения остальных.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

TRUNCATE PARTITION быстрее DELETE всех строк.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_19_truncate_partition здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',19);

### Задание 20. `m_razhin.gps_20_split_default`

**Что сделать:** Перенесите диапазон из default в явную partition.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Используйте SPLIT DEFAULT PARTITION.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_20_split_default здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',20);

## Уровень 3 — Direct Partition Exchange

### Задание 21. `m_razhin.gps_21_staging_like`

**Что сделать:** Создайте staging через LIKE целевой leaf-структуры.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Типы и physical options должны быть совместимы.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_21_staging_like здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',21);

### Задание 22. `m_razhin.gps_22_stage_latest`

**Что сделать:** Загрузите в staging данные последней даты источника.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

До exchange проверьте только нужный диапазон.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_22_stage_latest здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',22);

### Задание 23. `m_razhin.gps_23_change_key`

**Что сделать:** Измените event_date staging на следующий месяц.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Все строки должны попасть в границы новой partition.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_23_change_key здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',23);

### Задание 24. `m_razhin.gps_24_add_exchange_target`

**Что сделать:** Добавьте пустую target partition для нового месяца.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Сначала DDL, затем exchange.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_24_add_exchange_target здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',24);

### Задание 25. `m_razhin.gps_25_exchange_without_validation`

**Что сделать:** Выполните Direct Partition Exchange со staging.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

WITHOUT VALIDATION допустим только после своей проверки.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_25_exchange_without_validation здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',25);

### Задание 26. `m_razhin.gps_26_exchange_counts`

**Что сделать:** Создайте VIEW сверки строк до/после exchange.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Проверяйте target и бывшую leaf-таблицу.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_26_exchange_counts здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',26);

### Задание 27. `m_razhin.gps_27_exchange_metadata`

**Что сделать:** Докажите, что exchange меняет metadata, а не переписывает строки.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Сравните relfilenode/размер до и после.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_27_exchange_metadata здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',27);

### Задание 28. `m_razhin.gps_28_multilevel`

**Что сделать:** Создайте двухуровневую таблицу: месяц → list(country).

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Не создавайте чрезмерное число мелких leaf partitions.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_28_multilevel здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',28);

### Задание 29. `m_razhin.gps_29_partition_skew`

**Что сделать:** Создайте VIEW skew по каждой leaf partition.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Объедините partition tree и gp_segment_id counts.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_29_partition_skew здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',29);

### Задание 30. `m_razhin.gps_30_recommendation`

**Что сделать:** Создайте итоговый VIEW: объект, storage, compression, size, pruning, skew, verdict.

До выполнения запишите ожидаемый storage/partition effect. После — проверьте системный каталог, count и физический размер или план.

<details><summary>Подсказка</summary>

Рекомендация должна опираться на измерения.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте или измените m_razhin.gps_30_recommendation здесь.

In [ ]:
%%sql
-- Ручная проверка объекта, каталога, count или EXPLAIN.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('storage_partitioning',30);

## Прогресс

In [ ]:
%%sql
SELECT * FROM greenplum_training.progress WHERE module_name='storage_partitioning' ORDER BY task_no;